In [ ]:
import platform, subprocess
def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
print("hostname :", platform.node() or "n/a")
print("os       :", platform.platform())
print("nproc    :", _sh("nproc"))
print("cpu      :", _sh("grep -m1 -E 'model name' /proc/cpuinfo") or "no /proc/cpuinfo")
print("ram:")
print(_sh("free -g") or "free unavailable")


In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y --no-install-recommends r-base r-base-dev libcurl4-openssl-dev > /dev/null 2>&1
!R --version | head -1
print("R install done")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")


In [ ]:
import os, subprocess, urllib.request, hashlib

# Paste the URL + SHA256 printed by the P0-T4 bundle notebook (Cell 7).
BUNDLE_URL     = "PASTE_BUNDLE_URL"      # direct link to tisca_rlib.tar.gz
BUNDLE_SHA256  = "PASTE_BUNDLE_SHA256"   # Cell 7 of P0T4_build_rlib_bundle.ipynb
assert BUNDLE_URL != "PASTE_BUNDLE_URL", "paste BUNDLE_URL"
assert BUNDLE_SHA256 != "PASTE_BUNDLE_SHA256", "paste BUNDLE_SHA256"

DEST = "/content"
LIBDIR = "/content/tisca_rlib/rlib"
DL = "/content/_dl_tisca_rlib.tar.gz"
urllib.request.urlretrieve(BUNDLE_URL, DL)
h = hashlib.sha256(open(DL, "rb").read()).hexdigest()
assert h == BUNDLE_SHA256, "SHA mismatch"
if os.path.exists(LIBDIR):
    subprocess.run(["rm", "-rf", "/content/tisca_rlib"])
subprocess.run(["tar", "xzf", DL, "-C", DEST])
assert os.path.isdir(LIBDIR)
print("bundle restored:", LIBDIR)
print("tisca_rlib.rlib entries:", len(os.listdir(LIBDIR)))


In [ ]:
import os, subprocess, urllib.request

# run_cell.R lives in this repo; MVBCF_Code.cpp is the UPSTREAM file (the
# read-only NVBCF repo). We fetch both at runtime and never copy the .cpp
# into the TISCA repository (licensing/attribution rule, plan §0.1).
RUNCELL_URL = "https://raw.githubusercontent.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA/main/experiments/E3_mvbcf_casestudy/run_cell.R"
MVBCF_CPP_URL = "https://raw.githubusercontent.com/Nathan-McJames/MVBCF_Paper/main/MVBCF_Code.cpp"

os.makedirs("/content/e3", exist_ok=True)
urllib.request.urlretrieve(RUNCELL_URL, "/content/e3/run_cell.R")
urllib.request.urlretrieve(MVBCF_CPP_URL, "/content/e3/MVBCF_Code.cpp")
assert os.path.getsize("/content/e3/run_cell.R") > 1000
assert os.path.getsize("/content/e3/MVBCF_Code.cpp") > 10000
print("run_cell.R :", os.path.getsize("/content/e3/run_cell.R"), "bytes")
print("MVBCF_Code.cpp :", os.path.getsize("/content/e3/MVBCF_Code.cpp"), "bytes")


In [ ]:
# Compile the upstream MVBCF C++ once, verify fast_bart() is exposed, and
# confirm the bundle satisfies Rcpp + RcppArmadillo + RcppDist. Compilation
# happens once per session (not per replication).
import subprocess, os
compile_r = (
    '.libPaths(c("/content/tisca_rlib/rlib", .libPaths()))\n' +
    'if (!requireNamespace("Rcpp", quietly=TRUE) \n' +
    '    || !requireNamespace("RcppArmadillo", quietly=TRUE) \n' +
    '    || !requireNamespace("RcppDist", quietly=TRUE)) \n' +
    '  stop("bundle missing Rcpp/RcppArmadillo/RcppDist") \n' +
    'library(Rcpp) \n' +
    'sourceCpp("/content/e3/MVBCF_Code.cpp") \n' +
    'stopifnot(is.function(fast_bart)) \n' +
    'cat("FAST_BART_OK\\n") \n'
)
with open("/content/e3/compile.R", "w") as f:
    f.write(compile_r)
res = subprocess.run(["Rscript", "/content/e3/compile.R"], capture_output=True, text=True)
print(res.stdout[-3000:])
if res.returncode != 0 or "FAST_BART_OK" not in res.stdout:
    print("STDERR:", res.stderr[-3000:])
    raise SystemExit("Rcpp compilation of MVBCF_Code.cpp failed")
print("fast_bart() compiled and available")


## Round 0 pilot + P3-T5(e) stochtree::bcf calibration gate

Run **DGP1 n=500 pilot first** (the only cell the calibration gate needs).
It is ~5 core-hours. If the calibrated-BCF configuration is wrong, you lose
one notebook, not four. After an acceptable calibration result, launch the
remaining pilots (DGP2/DGP3 n=500, DGP1 n=100) in their own shards.

The gate compares the 50-pilot means of the calibrated `bcf_*` columns to
McJames et al.'s published DGP1/n=500 Table 2. No confirmatory shard may run
until all four BCF rows are inside their pass bands (plan P3-T5[e]).


In [ ]:
import os
# Round 0 pilot: DGP1, n=500, 50 independent-seed pilot replications.
DRIVE_DIR = "/content/drive/MyDrive/TISCA_E3"
DGP, N, MC_CORES, MODE = 1, 500, 2, "pilot"
SEED_START, SEED_END = 0, 49
os.makedirs(DRIVE_DIR, exist_ok=True)
out = os.path.join(DRIVE_DIR, f"DGP1_n500_pilot.csv")
import subprocess, time
env = dict(os.environ)
env["R_LIBS"] = LIBDIR + ":" + env.get("R_LIBS", "")
env["TISCA_MVBCF_CPP"] = "/content/e3/MVBCF_Code.cpp"
cmd = ["Rscript", "/content/e3/run_cell.R", str(DGP), str(N),
       str(SEED_START), str(SEED_END), "--out", out,
       "--cores", str(MC_CORES), "--mode", MODE]
print("running:", " ".join(cmd))
t0 = time.time()
res = subprocess.run(cmd, env=env, capture_output=True, text=True)
print("wall-clock: %.1f min" % ((time.time() - t0) / 60.0))
print(res.stdout[-4000:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-3000:])
    raise SystemExit("pilot failed")


In [ ]:
import csv, statistics as st
with open(out) as f:
    rows = list(csv.DictReader(f))
n = len(rows)
if n < 30:
    print("[FAIL] calibration needs >=50 pilot rows; got", n)
f1 = lambda key: [float(r[key]) for r in rows if r.get(key) not in (None, "")]
res = {}
for key in ["bcf_pehe1", "bcf_pehe2", "bcf_cov951", "bcf_cov952"]:
    v = f1(key)
    res[key] = (st.mean(v), st.stdev(v) / (len(v) ** 0.5), len(v)) if v else (float("nan"), float("nan"), 0)
print("=" * 72)
print("P3-T5(e) calibration gate vs McJames et al. DGP1 n=500 Table 2")
print("=" * 72)
targets = {
    "bcf_pehe1": (9.63, 9.30, 10.00, "BCF PEHE Y1"),
    "bcf_pehe2": (9.96, 9.60, 10.30, "BCF PEHE Y2"),
    "bcf_cov951": (0.97, 0.95, 0.98, "BCF tau 95% cov Y1"),
    "bcf_cov952": (0.96, 0.94, 0.98, "BCF tau 95% cov Y2"),
}
ok = True
for key, (mean, se, nn) in res.items():
    tgt, lo, hi, label = targets[key]
    inside = lo <= mean <= hi
    ok = ok and inside
    print(f"{label:>20}: mean={mean:.3f} se={se:.3f} target={tgt} band=[{lo},{hi}] -> {'OK' if inside else 'OUT OF BAND'}")
print("-" * 72)
print("VERDICT:", "PASS - proceed to confirmatory shards" if ok
      else "FAIL - diagnose before launching confirmatory (plan P3-T5[e])")


In [ ]:
# P3-T5(f): compare fast_bart() vs mvbcf::run_mvbcf() on 10 identical seeds.
# Placeholder: run both on the same 10 pilot seeds and compare PEHE/coverage.
print("P3-T5(f) fast_bart-vs-mvbcf equivalence check: "
      "see CALIBRATION.md; record outcome there.")
